# Fine-tuning Qwen 3.5 27B
Can parameter count fix all my problems?

In [ ]:
%load_ext autoreload
%autoreload 2

import os
os.environ["HF_HUB_OFFLINE"] = "1"

from dotenv import load_dotenv
load_dotenv()
SYSTEM_PROMPT = os.getenv("SYSTEM_PROMPT") 

import torch
from datasets import load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

from evaluation import formatting_func

## Model + Data Prep

In [ ]:
max_seq_length = 2048
model_name = "eb-qwen-27b-lora-522"

model, tokenizer= FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3.5-27B",
    max_seq_length = max_seq_length,
    load_in_4bit = False,
    full_finetuning = False,
    dtype = torch.bfloat16,
    device_map= "auto",
)

In [ ]:
train_path = "../data/echobot_old/train.jsonl"
val_path = "../data/echobot_old/val.jsonl"
test_path = "../data/echobot_old/test.jsonl"

dataset = load_dataset(
    "json",
    data_files={
        "train": train_path,
        "validation": val_path,
        "test": test_path,
    },
)
dataset["train"] = dataset["train"].shuffle(seed=8)
print(dataset['train'].features)

In [ ]:
from collections import Counter

# format dataset with only 'prompt' and 'completion' collumns
formatted_train = dataset["train"].map(
    formatting_func,
    remove_columns=dataset["train"].column_names
    )
formatted_val = dataset["validation"].map(
    formatting_func,
    remove_columns=dataset["validation"].column_names
    )
formatted_test = dataset["test"].map(
    formatting_func,
    remove_columns=dataset["test"].column_names
    )

print(Counter(formatted_train["completion"]))
print(Counter(formatted_val["completion"]))
print(Counter(formatted_test["completion"]))


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 10,
    lora_alpha = 20,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        # "gate_proj", "up_proj", "down_proj",
    ],

    finetune_vision_layers = False,
    finetune_language_layers = True, 
    finetune_attention_modules = True,
    finetune_mlp_modules = True,

    lora_dropout = 0.10,
    bias = "none",
    use_gradient_checkpointing = "unsloth",

    random_state = 8,
    max_seq_length = max_seq_length,
)

## Training

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_train,
    eval_dataset=formatted_val,
    processing_class=tokenizer,

    args=SFTConfig(
        max_seq_length=max_seq_length,
        completion_only_loss=True,
        per_device_train_batch_size=8,

        eval_strategy="steps",
        gradient_accumulation_steps=4,
        warmup_ratio=0.05,
        num_train_epochs=3,
        optim="adamw_8bit",

        seed=8,
        dataset_num_proc=1,

        output_dir=f"../checkpoints/{model_name}",
        logging_steps=1,
    ),
)
trainer.train()

## Evaluation

In [ ]:
from evaluation import (
    evaluate_model, 
    plot_loss
)
plot_loss(trainer=trainer, show_val=True)

evaluate_model(
    model=model, 
    tokenizer=tokenizer,
    dataset=dataset['test'],
    model_str=model_name,
    batch_size=16
)

# Save model adapters to disk 

In [ ]:
adapter_path = f"../adapters/{model_name}"

model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print(f"Saved LoRA adapter to {adapter_path}")